<a href="https://colab.research.google.com/github/shinobu357/TugasMLRaisya/blob/main/Week%2014/Week_14__Raisya_Athaya_Kamilah_101032380253_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# Import Libraries
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

# Memuat dataset
data = pd.read_csv('/content/drive/MyDrive/Week 11/heart.csv')  # Ganti dengan path file Anda

# Tampilkan beberapa baris pertama dataset untuk memeriksa struktur data
print(data.head())

# Misalkan kita ingin memisahkan fitur dan label (sesuaikan dengan dataset Anda)
X = data.drop('target', axis=1).values  # Ganti dengan nama kolom target
y = data['target'].values  # Ganti dengan nama kolom target

# Normalisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Membagi data menjadi training dan testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Mengonversi ke tensor
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Membuat DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)




   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  
3   1     3       0  
4   3     2       0  


In [20]:
# Definisi Model RNN Sederhana
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)

        # Memeriksa dimensi output dari RNN
        if out.dim() == 2:  # Jika hanya ada satu timestep
            out = out  # Ambil output langsung
        else:
            out = out[:, -1, :]  # Ambil output dari timestep terakhir

        out = self.fc(out)
        return out

# Definisi Model Deep RNN
class DeepRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=2):
        super(DeepRNN, self).__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)

        # Menangani kasus di mana output hanya 2D
        if out.dim() == 2:  # Ketika hanya ada satu timestep
            out = out  # Ambil output langsung
        else:
            out = out[:, -1, :]  # Ambil output dari timestep terakhir

        out = self.fc(out)
        return out


In [21]:
# Fungsi untuk melatih model
def train_model(model, train_loader, test_loader, optimizer, criterion, num_epochs, scheduler=None):
    best_model = None
    best_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # Menggunakan scheduler jika ada
        if scheduler:
            scheduler.step()

        # Menghitung average loss
        avg_loss = running_loss / len(train_loader)

        # Evaluasi pada data testing
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total * 100
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

        # Simpan model terbaik berdasarkan loss
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_model = model.state_dict()

    return best_model

# Fungsi untuk menjalankan eksperimen
def run_experiments():
    # Hyperparameters
    input_size = X_train_tensor.shape[1]
    hidden_sizes = [32, 64, 128]  # Misalkan kita mencoba beberapa ukuran hidden layer
    num_epochs_list = [5, 50, 100, 250, 350]
    optimizers = [optim.SGD, optim.RMSprop, optim.Adam]

    best_models = {}

    for hidden_size in hidden_sizes:
        for num_epochs in num_epochs_list:
            for optimizer_func in optimizers:
                print(f"\nTraining with hidden_size={hidden_size}, epochs={num_epochs}, optimizer={optimizer_func.__name__}")

                # Model
                model = SimpleRNN(input_size=input_size, hidden_size=hidden_size, output_size=2)  # Ganti output_size jika perlu

                # Loss function dan optimizer
                criterion = nn.CrossEntropyLoss()
                optimizer = optimizer_func(model.parameters(), lr=0.001)
                scheduler = StepLR(optimizer, step_size=50, gamma=0.1)

                # Melatih model
                best_model = train_model(model, train_loader, test_loader, optimizer, criterion, num_epochs, scheduler)

                # Simpan model terbaik
                best_models[(hidden_size, num_epochs, optimizer_func.__name__)] = best_model

    return best_models

# Jalankan eksperimen
best_models = run_experiments()

Streaming output truncated to the last 5000 lines.
Epoch [293/350], Loss: 0.2931, Accuracy: 80.49%
Epoch [294/350], Loss: 0.2943, Accuracy: 80.49%
Epoch [295/350], Loss: 0.2936, Accuracy: 80.49%
Epoch [296/350], Loss: 0.2957, Accuracy: 80.49%
Epoch [297/350], Loss: 0.2937, Accuracy: 80.49%
Epoch [298/350], Loss: 0.2939, Accuracy: 80.49%
Epoch [299/350], Loss: 0.2940, Accuracy: 80.49%
Epoch [300/350], Loss: 0.2929, Accuracy: 80.49%
Epoch [301/350], Loss: 0.2928, Accuracy: 80.49%
Epoch [302/350], Loss: 0.2946, Accuracy: 80.49%
Epoch [303/350], Loss: 0.2944, Accuracy: 80.49%
Epoch [304/350], Loss: 0.2914, Accuracy: 80.49%
Epoch [305/350], Loss: 0.2932, Accuracy: 80.49%
Epoch [306/350], Loss: 0.2923, Accuracy: 80.49%
Epoch [307/350], Loss: 0.2938, Accuracy: 80.49%
Epoch [308/350], Loss: 0.2931, Accuracy: 80.49%
Epoch [309/350], Loss: 0.2972, Accuracy: 80.49%
Epoch [310/350], Loss: 0.2913, Accuracy: 80.49%
Epoch [311/350], Loss: 0.2934, Accuracy: 80.49%
Epoch [312/350], Loss: 0.2960, Accura